# OpenAI 음성/언어 모델 테스트 노트북

이 노트북은 Azure OpenAI 의 다음 모델들을 빠르게 검증하기 위한 예제 모음이다.

| 용도 | 모델/배포명 |
|------|------|
| STT (음성 → 텍스트) | `gpt-4o-mini-transcribe` |
| TTS (텍스트 → 음성) | `gpt-4o-mini-tts` |
| LLM (텍스트 → 텍스트) | `gpt-5.4-mini` |
| Realtime (음성 ↔ 음성/텍스트, WebSocket) | `gpt-realtime-mini` |

---

## 환경 셋업 (최초 1회)

이 노트북은 conda env `voice_ai_practices` 에 등록된 ipykernel 을 사용한다.

```bash
# 1) UV + 의존성 설치 + ipykernel 등록까지 한 번에
bash setup_env.sh
```

`setup_env.sh` 가 하는 일:

1. `voice_ai_practices` conda env (Python 3.10) 생성/활성화
2. `uv` 설치 (없으면)
3. `uv pip sync requirements.txt` 로 의존성 동기화
4. `ipykernel` 으로 커널 `voice_ai_practices` 등록

> **requirements.txt 갱신이 필요하면** 먼저 아래를 직접 실행한다.
> ```bash
> uv pip compile requirements.in -o requirements.txt --python-version 3.10
> ```

이 노트북의 `metadata.kernelspec.name` 도 `voice_ai_practices` 로 설정되어 있어,
Jupyter 가 자동으로 같은 이름의 커널을 잡아준다 (수동 변경: `Kernel > Change Kernel > voice_ai_practices`).

---

## .env 설정

Azure OpenAI 자격증명은 같은 디렉토리의 `.env` 에서 로드한다.

```
AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com/
AZURE_OPENAI_API_KEY=<your-key>
```


## 0. 공통 설정

`.env` 를 로드하고 Azure OpenAI 동기 클라이언트를 생성한다.
배포명(deployment name)은 Azure 포털에서 만든 이름과 정확히 일치해야 한다.

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

# Azure OpenAI 공통 클라이언트
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version="2025-04-01-preview",
)

# Azure 배포명 (포털에서 만든 deployment 이름과 동일해야 함)
AZURE_STT_DEPLOYMENT = "gpt-4o-mini-transcribe"
AZURE_TTS_DEPLOYMENT = "gpt-4o-mini-tts"
AZURE_LLM_DEPLOYMENT = "gpt-5.4-mini"
AZURE_REALTIME_DEPLOYMENT = "gpt-realtime-mini"

print("endpoint :", os.getenv("AZURE_OPENAI_ENDPOINT"))
print("api key  :", "set" if os.getenv("AZURE_OPENAI_API_KEY") else "MISSING")

endpoint : https://gpt-team-datascience-east-us-2.openai.azure.com/
api key  : set


## 1. STT — `gpt-4o-mini-transcribe`

음성 파일(`.wav`/`.mp3` 등)을 한국어 텍스트로 변환한다.

- `model` 파라미터에는 **모델명이 아니라 Azure 배포명**을 넣어야 한다.
- `response_format="text"` 로 설정하면 결과가 단순 문자열로 반환된다 (JSON 파싱 불필요).
- `language="ko"` 힌트를 주면 한국어 인식 정확도가 올라간다.

In [4]:
def transcribe_korean_azure(audio_path: str) -> str:
    """Azure OpenAI STT (gpt-4o-mini-transcribe)."""
    if not AZURE_STT_DEPLOYMENT:
        raise ValueError("AZURE_STT_DEPLOYMENT 가 비어있습니다.")

    with open(audio_path, "rb") as audio_file:
        result = client.audio.transcriptions.create(
            model=AZURE_STT_DEPLOYMENT,
            file=audio_file,
            language="ko",
            response_format="text",
        )
    return result


# 실행 예시: 같은 디렉토리에 sample_ko.m4a 파일을 둔다.
audio_file = "sample_ko.m4a"
if Path(audio_file).exists():
    transcript = transcribe_korean_azure(audio_file)
    print("[Azure STT 결과]")
    print(transcript)
else:
    print(f"(스킵) {audio_file} 파일이 없습니다. 테스트하려면 동일 경로에 파일을 두세요.")

[Azure STT 결과]
안녕, 나는 배달의 민족에서 데이터 사이언티스트로 일하고 있는 김동완이야.



## 2. TTS — `gpt-4o-mini-tts`

텍스트를 한국어 음성(MP3)으로 합성한다.

- `voice` 는 `alloy`, `nova`, `shimmer`, `echo`, `fable`, `onyx` 중 선택. 톤이 모두 다르므로 직접 들어보고 고르면 된다.
- `instructions` 로 발화 스타일을 자연어로 지정할 수 있다 (예: "차분한 안내 톤", "활기차게").
- `with_streaming_response.create(...)` + `stream_to_file(...)` 패턴으로 메모리 사용을 최소화한다.

In [3]:
def synthesize_korean_azure(text: str, output_path: str = "azure_korean_tts.mp3") -> Path:
    """Azure OpenAI TTS (gpt-4o-mini-tts).

    `instructions` 는 openai SDK >= 1.70 에서만 지원되는 발화 스타일 제어 파라미터.
    requirements.in 의 openai 버전이 그 이하면 이 인자를 빼야 한다.
    """
    if not AZURE_TTS_DEPLOYMENT:
        raise ValueError("AZURE_TTS_DEPLOYMENT 가 비어있습니다.")

    output_file = Path(output_path)

    with client.audio.speech.with_streaming_response.create(
        model=AZURE_TTS_DEPLOYMENT,
        voice="alloy",
        input=text,
        instructions="한국어를 자연스럽고 명확하게, 친절한 안내 톤으로 읽어주세요.",
        response_format="mp3",
    ) as response:
        response.stream_to_file(output_file)

    return output_file


out = synthesize_korean_azure(
    "안녕하세요. 한국어 음성 합성 테스트입니다.",
    "azure_korean_tts.mp3",
)
print(f"[Azure TTS 결과] {out.resolve()}")

[Azure TTS 결과] /Users/donghwan.kim/PycharmProjects/nlp_experiments/voice_ai_practices/azure_korean_tts.mp3


## 3. LLM — `gpt-5.4-mini`

일반 챗 컴플리션 (텍스트 → 텍스트) 호출.

- 위 공통 설정에서 만든 `client` 와 `.env` 의 `AZURE_OPENAI_*` 를 그대로 사용한다 (별도 endpoint/key 하드코딩 불필요).
- `max_completion_tokens` 는 모델 출력의 상한을 지정한다 (Azure 신규 모델 계열은 `max_tokens` 대신 이 파라미터 사용).
- `model` 자리에 **Azure 배포명**을 넣는다.

In [5]:
def chat_complete(prompt: str, system: str = "You are a helpful assistant.") -> str:
    """Azure OpenAI LLM (gpt-5.4-mini)."""
    response = client.chat.completions.create(
        model=AZURE_LLM_DEPLOYMENT,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        max_completion_tokens=1024,
    )
    return response.choices[0].message.content


answer = chat_complete("파리에 가면 뭘 봐야 해? 핵심 3개만 짧게 알려줘.")
print("[Azure LLM 결과]")
print(answer)

[Azure LLM 결과]
파리에 가면 핵심 3개만 보면 돼요:

1. **에펠탑** – 파리의 상징, 야경까지 보면 가장 좋음  
2. **루브르 박물관** – 모나리자, 고전 명작들  
3. **세느강 산책/유람선** – 도시 분위기를 가장 잘 느낄 수 있음

원하면 “반나절 코스”로도 딱 짜드릴게요.


## 4. Realtime — `gpt-realtime-mini`

WebSocket 으로 양방향 스트리밍하는 실시간 음성/텍스트 모델.
이 셀은 **연결 → 텍스트 입력 → 텍스트+오디오 응답 수신** 까지의 최소 동작을 검증한다.

핵심 포인트:

- `AsyncAzureOpenAI.beta.realtime.connect(model=<배포명>)` 로 WebSocket 세션 시작.
- `session.update(...)` 로 modality(`text` / `audio`), 음성, 입력 오디오 포맷 등을 설정.
- `conversation.item.create(...)` 로 사용자 입력을 추가하고 `response.create()` 로 응답 생성을 트리거.
- 이벤트 스트림(`async for event in connection`)에서 `response.text.delta`, `response.audio.delta`, `response.done` 등을 처리.

> Jupyter 는 이미 이벤트 루프가 돌고 있으므로 `asyncio.run` 대신 `await` 또는 `nest_asyncio` 를 사용한다.
> 아래에서는 `await main()` 한 줄로 실행한다.

> 마이크/스피커로 풀 양방향 음성을 하려면 `sounddevice` 로 PCM16 24kHz 청크를
> `input_audio_buffer.append` 이벤트로 보내고, 응답 `response.audio.delta` 를 재생하면 된다.
> 여기서는 텍스트 입력 → 텍스트+오디오 응답으로 단순화한다.

In [6]:
import asyncio
import base64
from openai import AsyncAzureOpenAI

# Realtime 은 별도 async 클라이언트 인스턴스를 쓴다.
async_client = AsyncAzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version="2025-04-01-preview",
)


async def realtime_text_to_speech_demo(user_text: str, out_audio_path: str = "realtime_response.pcm") -> dict:
    """gpt-realtime-mini 에 텍스트를 보내고 텍스트+오디오 응답을 받는다.

    반환:
        {"text": <누적 텍스트>, "audio_path": <PCM16 24kHz mono 파일 경로 또는 None>}
    """
    audio_chunks: list[bytes] = []
    text_parts: list[str] = []

    async with async_client.beta.realtime.connect(
        model=AZURE_REALTIME_DEPLOYMENT,
    ) as connection:
        # 세션 설정: 텍스트+오디오 출력, 한국어 친화 음성
        await connection.session.update(session={
            "modalities": ["text", "audio"],
            "voice": "alloy",
            "instructions": "한국어로 자연스럽고 친절하게 짧게 답해주세요.",
            "output_audio_format": "pcm16",
        })

        # 사용자 메시지 추가
        await connection.conversation.item.create(item={
            "type": "message",
            "role": "user",
            "content": [{"type": "input_text", "text": user_text}],
        })

        # 응답 생성 트리거
        await connection.response.create()

        # 이벤트 스트림 처리
        async for event in connection:
            etype = event.type
            if etype == "response.text.delta":
                text_parts.append(event.delta)
                print(event.delta, end="", flush=True)
            elif etype == "response.audio.delta":
                # delta 는 base64 인코딩된 PCM16 24kHz mono
                audio_chunks.append(base64.b64decode(event.delta))
            elif etype == "response.done":
                print()  # 텍스트 출력 줄바꿈
                break
            elif etype == "error":
                print(f"\n[ERROR] {event}")
                break

    audio_path = None
    if audio_chunks:
        audio_path = Path(out_audio_path)
        audio_path.write_bytes(b"".join(audio_chunks))

    return {"text": "".join(text_parts), "audio_path": str(audio_path) if audio_path else None}


# Jupyter 안에서는 await 로 직접 실행
result = await realtime_text_to_speech_demo("자기소개를 한 문장으로 해줘.")
print("\n[Realtime 결과 텍스트]", result["text"])
print("[Realtime 응답 오디오]", result["audio_path"], "(PCM16, 24kHz, mono)")



[Realtime 결과 텍스트] 
[Realtime 응답 오디오] realtime_response.pcm (PCM16, 24kHz, mono)


### Realtime 응답 오디오 재생 (선택)

저장된 PCM16/24kHz mono raw 파일은 그대로는 대부분의 플레이어에서 안 열린다.
WAV 헤더를 붙이거나 `sounddevice` 로 직접 재생한다.

In [7]:
import wave

def pcm16_to_wav(pcm_path: str, wav_path: str, sample_rate: int = 24000) -> str:
    pcm = Path(pcm_path).read_bytes()
    with wave.open(wav_path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)  # 16-bit
        wf.setframerate(sample_rate)
        wf.writeframes(pcm)
    return wav_path


if result["audio_path"]:
    wav = pcm16_to_wav(result["audio_path"], "realtime_response.wav")
    print(f"[WAV 변환 완료] {Path(wav).resolve()}")

    # Jupyter 인라인 재생
    from IPython.display import Audio, display
    display(Audio(wav))

[WAV 변환 완료] /Users/donghwan.kim/PycharmProjects/nlp_experiments/voice_ai_practices/realtime_response.wav
